<a href="https://colab.research.google.com/github/amaimanwar8-arch/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/amaimanwar8-arch/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("Working dir:", os.getcwd())
print(df.shape[0], "pages loaded successfully.")

Working dir: /content/flyrank-ml-internship
30000 pages loaded successfully.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [2]:
print("""
Finding A - "What Predicts Health?" (Random Forest, ML Appendix, page 27)
Claim: Average Position (43%) and Impressions (32%) are the top predictors of health_score.

My methodology question: health_score is explicitly defined elsewhere in this same paper
as Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts) - so
avg_position and impressions are literally INGREDIENTS of the target, not independent
predictors of it. The paper does flag this honestly ("importance is descriptive rather
than causal"), which I respect - but I'd ask: why report it as "feature importance" at
all, rather than naming it directly as a circular/leakage result the way Notebook 04's
leakage trap did? A reader skimming the bar chart without reading the caveat text could
easily walk away thinking these are genuine external predictors.

Finding B - "What Predicts Growth?" (Logistic Regression, ML Appendix, page 29)
Claim: 71% holdout accuracy separating growing vs. declining pages.

My methodology question: the paper doesn't say HOW the holdout was built - random split,
or grouped by brand? This matters a lot, because my own Week-5 notebook found a random
split can inflate results: my random forest scored well on a random split but LOST to a
simple baseline once I switched to a client-grouped split (Precision@50: 0.760 model vs
0.840 baseline). If this 71% number came from a random split across 57 brands, the same
brand could appear in both train and holdout, and the real number on genuinely unseen
brands could be lower. I'd ask this respectfully, the same way I'd want my own Week-5
number checked - it's a fair methodology question, not a claim that the paper is wrong.
""")


Finding A - "What Predicts Health?" (Random Forest, ML Appendix, page 27)
Claim: Average Position (43%) and Impressions (32%) are the top predictors of health_score.

My methodology question: health_score is explicitly defined elsewhere in this same paper
as Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts) - so
avg_position and impressions are literally INGREDIENTS of the target, not independent
predictors of it. The paper does flag this honestly ("importance is descriptive rather
than causal"), which I respect - but I'd ask: why report it as "feature importance" at
all, rather than naming it directly as a circular/leakage result the way Notebook 04's
leakage trap did? A reader skimming the bar chart without reading the caveat text could
easily walk away thinking these are genuine external predictors.

Finding B - "What Predicts Growth?" (Logistic Regression, ML Appendix, page 29)
Claim: 71% holdout accuracy separating growing vs. declining pages.

My method

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

visible = df[df["impressions_90d"] >= 100].copy()
model_features = ["content_age_days", "days_since_last_update", "impressions_90d",
                   "avg_position", "ctr", "word_count"]
visible[model_features] = visible[model_features].replace([np.inf, -np.inf], np.nan).fillna(0)
X = visible[model_features]
y = visible["is_declining_label"]

# --- BEFORE: random split (no grouping) ---
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model_r = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1).fit(X_tr_r, y_tr_r)
score_r = model_r.predict_proba(X_te_r)[:, 1]

# --- AFTER: client-grouped split (same as Week 5) ---
groups = visible["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]
model_g = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1).fit(X_tr_g, y_tr_g)
score_g = model_g.predict_proba(X_te_g)[:, 1]

before_after = pd.DataFrame([
    {"split": "random (BEFORE)", "precision@20": round(precision_at_k(score_r, y_te_r.values, 20), 3),
     "precision@50": round(precision_at_k(score_r, y_te_r.values, 50), 3)},
    {"split": "client-grouped (AFTER)", "precision@20": round(precision_at_k(score_g, y_te_g.values, 20), 3),
     "precision@50": round(precision_at_k(score_g, y_te_g.values, 50), 3)},
])
print("Before/after: same model, same features, only the split design changes.")
before_after

Before/after: same model, same features, only the split design changes.


,split,precision@20,precision@50
0,random (BEFORE),1.00,0.98
1,client-grouped (AFTER),0.65,0.76


In [4]:
print("""
Before/after result: the SAME model, SAME features, only the split design changed.

Random split (BEFORE): Precision@20 = 1.000, Precision@50 = 0.980 - looks almost perfect.
Client-grouped split (AFTER): Precision@20 = 0.650, Precision@50 = 0.760 - a real, honest
number, and notably WORSE than my Week-4 baseline (0.850 / 0.840) under the same grouped
split.

This is the clearest demonstration I've built yet of why split design matters more than
which model you pick. A near-perfect 1.00/0.98 was never real skill - it was the model
partly memorizing per-client patterns it then saw again in "test" rows from the same
clients. The moment clients are genuinely held out, that illusion disappears completely.
This is exactly the methodology question I raised about Finding B in Section 1 - an
unspecified holdout design can hide this same gap, and I now have direct proof of how
large that gap can be on my own data.
""")


Before/after result: the SAME model, SAME features, only the split design changed.

Random split (BEFORE): Precision@20 = 1.000, Precision@50 = 0.980 - looks almost perfect.
Client-grouped split (AFTER): Precision@20 = 0.650, Precision@50 = 0.760 - a real, honest
number, and notably WORSE than my Week-4 baseline (0.850 / 0.840) under the same grouped
split.

This is the clearest demonstration I've built yet of why split design matters more than
which model you pick. A near-perfect 1.00/0.98 was never real skill - it was the model
partly memorizing per-client patterns it then saw again in "test" rows from the same
clients. The moment clients are genuinely held out, that illusion disappears completely.
This is exactly the methodology question I raised about Finding B in Section 1 - an
unspecified holdout design can hide this same gap, and I now have direct proof of how
large that gap can be on my own data.



## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
# Deliberate leak, same pattern as Week 3/04's trap - this time on the real model features
leak_test_features = model_features + ["trend_pct"]  # trend_pct is what is_declining_label is built from
X_leak = visible[leak_test_features].replace([np.inf, -np.inf], np.nan).fillna(0)

X_tr_l, X_te_l = X_leak.iloc[train_idx], X_leak.iloc[test_idx]
model_l = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1).fit(X_tr_l, y_tr_g)
score_l = model_l.predict_proba(X_te_l)[:, 1]

leak_p50 = precision_at_k(score_l, y_te_g.values, 50)
honest_p50 = precision_at_k(score_g, y_te_g.values, 50)

print(f"Honest model (6 features, client-grouped): Precision@50 = {honest_p50:.3f}")
print(f"With trend_pct added back in: Precision@50 = {leak_p50:.3f}  <- inflated by leakage")
print(f"\nConfirmed: {model_features} are all observed-before-decision-point signals, safe to use.")
print("trend_pct is excluded from my final feature set because it is the exact column")
print("is_declining_label (via trend_direction) is derived from - this was flagged as early")
print("as ML-03 and re-confirmed here on my final Week-5/06 feature set.")

Honest model (6 features, client-grouped): Precision@50 = 0.760
With trend_pct added back in: Precision@50 = 1.000  <- inflated by leakage

Confirmed: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count'] are all observed-before-decision-point signals, safe to use.
trend_pct is excluded from my final feature set because it is the exact column
is_declining_label (via trend_direction) is derived from - this was flagged as early
as ML-03 and re-confirmed here on my final Week-5/06 feature set.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [6]:
print("""
My boldest original claim (from ML-08 / Week-5 Section 1):
"A tree ensemble can weigh many of them together without needing careful feature scaling"
- combined with the ORIGINAL headline number I nearly reported before checking further:
"random forest reached Precision@50 = 0.740 vs. baseline's 0.240" (from Notebook 2,
in-sample/random split).

Rewritten in safe language:
"Under an IN-SAMPLE, random-split comparison, a random forest OBSERVED a higher
Precision@50 (0.740) than a hand-written baseline (0.240) on this labeled task. This
result is DIRECTIONAL evidence that ensemble methods can capture more signal than a short
rule - but it is NOT decision-support on its own: under a client-grouped holdout, the same
model MEASURED a lower Precision@50 (0.760) than the baseline (0.840) on genuinely unseen
clients. The honest, decision-support-grade conclusion is that my current 6-feature random
forest has not yet demonstrated it beats a simple, well-reasoned rule for new clients -
this is an observed limitation of my current feature set, not proof that ML never helps
here."

Why the rewrite matters: the original framing let a flashy in-sample number stand in for
real-world performance. The rewrite is longer and less exciting, but it's the version
that wouldn't mislead a content team into trusting a model that currently underperforms
their own simple rule on new clients.
""")


My boldest original claim (from ML-08 / Week-5 Section 1):
"A tree ensemble can weigh many of them together without needing careful feature scaling"
- combined with the ORIGINAL headline number I nearly reported before checking further:
"random forest reached Precision@50 = 0.740 vs. baseline's 0.240" (from Notebook 2,
in-sample/random split).

Rewritten in safe language:
"Under an IN-SAMPLE, random-split comparison, a random forest OBSERVED a higher
Precision@50 (0.740) than a hand-written baseline (0.240) on this labeled task. This
result is DIRECTIONAL evidence that ensemble methods can capture more signal than a short
rule - but it is NOT decision-support on its own: under a client-grouped holdout, the same
model MEASURED a lower Precision@50 (0.760) than the baseline (0.840) on genuinely unseen
clients. The honest, decision-support-grade conclusion is that my current 6-feature random
forest has not yet demonstrated it beats a simple, well-reasoned rule for new clients -
this is a

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.